[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/hubbard_ring/hubbard_ring.ipynb)

# The Hubbard Ring

A state of four electrons in eight orbitals is the wedge of the four occupied ones, `mv.a ^ mv.c ^ mv.e ^ mv.h`: it vanishes when two share an orbital and flips sign when two trade places. A one-electron map acts on all four electrons by one line, `(one_electron ^ (State & Complement)).trace(1, 3)`. The join with the open complement takes one occupied orbital out, the map's image is wedged back in its place, and the trace pairs the map's input with the orbital taken out. Nothing in the line counts electrons: the [Hubbard dimer](../hubbard_dimer/hubbard_dimer.ipynb) lifts its hop to two electrons by the same line. Lifted this way, hopping around a ring of four sites is a map on 4-vectors, and each orbital's projector becomes the number of electrons in that orbital.

When sharing a site is expensive, what is left are the spins: the ring's levels approach those of four Heisenberg spins, and a single flipped spin walks around the ring and arrives almost whole at the opposite site.

A flipped spin crossing a square of spins whole is an instance of the perfect state transfer of Christandl, Datta, Ekert and Landahl (Physical Review Letters 92, 187902, 2004). In the notation of second quantization the lift of a one-electron map $h$ reads as $\sum_{pq} h_{pq}\, \hat c^\dagger_p \hat c_q$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.quantum.hubbard_ring import render

np.set_printoptions(precision=3, suppress=True)

# Eight orbitals, spin up and spin down on each of four sites.
ga = Algebra("a+b+c+d+e+f+g+h+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Orbital = ga.gatype.vector()
State = ga.gatype(ga.subspace.k_vector(4))                                  # four electrons
Complement = ga.gatype.antivector()

# The orbitals by site and spin, up then down, and each site's neighbour along the ring.
orbitals = stack([stack([mv.a, mv.b]), stack([mv.c, mv.d]), stack([mv.e, mv.f]), stack([mv.g, mv.h])])   # [sites, spins] Orbital
following = stack([orbitals[1], orbitals[2], orbitals[3], orbitals[0]])     # [sites, spins] Orbital

# Energies use the hopping strength as their unit.
hopping_strength = 1.0

## 1. Four electrons, the same lift

An orbital is a vector, and four occupied orbitals form a 4-vector: a repeated orbital gives zero, and exchanging two changes the sign. A one-electron map acts on each of the four in turn and adds the results, its derivation extension to four electrons. The join with an open complement removes one occupied orbital, wedging the map's image back in replaces it, and the trace pairs the map's input with the complementary slot: the same line as for two electrons, now on 4-vectors. Hopping lifts that way. So does each orbital's projector, which becomes the number of electrons in that orbital; on each site the up number composed with the down number is one when the site holds both electrons and zero otherwise, and their sum over the sites counts the shared sites.

In the notation of second quantization the lifted hopping reads as $\hat T = -t\sum_{i\sigma}(\hat c^\dagger_{i+1,\sigma}\hat c_{i\sigma} + \hat c^\dagger_{i\sigma}\hat c_{i+1,\sigma})$, and the shared sites as $\sum_i \hat n_{i\uparrow}\hat n_{i\downarrow}$.

In [ ]:
def lift(one_electron):
    """A one-electron map acting on each of the four electrons in turn."""
    return (one_electron ^ (State & Complement)).trace(1, 3)                 # [...] State <- State


# Each dyad reads an orbital on one site and returns the same spin on the next, and back.
single_hopping = -hopping_strength * (following * (orbitals | Orbital) + orbitals * (following | Orbital)).sum(axis=0).sum(axis=0)   # [] Orbital <- Orbital
hopping = lift(single_hopping)                                               # [] State <- State
# Each orbital's projector, lifted: how many electrons the orbital holds.
numbers = lift(orbitals * (orbitals | Orbital))                              # [sites, spins] State <- State
shared_sites = numbers[:, 0](numbers[:, 1]).sum(axis=0)                      # [] State <- State

# Spin up on the first three sites and spin down on the last; the lift, against the four terms.
start = mv.a ^ mv.c ^ mv.e ^ mv.h                                            # [] State
each_in_turn = ((single_hopping(mv.a) ^ mv.c ^ mv.e ^ mv.h) + (mv.a ^ single_hopping(mv.c) ^ mv.e ^ mv.h)
                + (mv.a ^ mv.c ^ single_hopping(mv.e) ^ mv.h) + (mv.a ^ mv.c ^ mv.e ^ single_hopping(mv.h)))   # [] State

In [ ]:
print("lifted hopping less each electron hopping in turn:", np.abs((hopping(start) - each_in_turn).kernel).max())
print("shared sites of the start, and of both electrons on the first two sites:",
      start.reverse().scalar_product(shared_sites(start)).to_array(),
      (mv.a ^ mv.b ^ mv.c ^ mv.d).reverse().scalar_product(shared_sites(mv.a ^ mv.b ^ mv.c ^ mv.d)).to_array())

## 2. Charge motion gives way to spins

With the repulsion for sharing a site, the Hamiltonian is the lifted hopping plus the repulsion times the shared sites, a map on 4-vectors, diagonalized. One electron per site leaves sixteen ways to arrange the spins, and as the repulsion grows these sixteen levels separate from the rest and close in on each other: hopping survives only as brief visits to a neighbour and back, which exchange the two spins at the rate four times the hopping squared over the repulsion. In units of that exchange the sixteen levels settle where the Heisenberg ring of four spins puts them above its ground state: three at one, seven at two, five at three. Below, those levels against the repulsion, the Heisenberg ring's dashed.

In the notation of spin models the limit reads as the Heisenberg ring $J\sum_i \mathbf S_i\cdot\mathbf S_{i+1}$ with $J = 4t^2/U$, whose levels $\tfrac J2\left[S(S+1) - S_A(S_A+1) - S_B(S_B+1)\right]$ follow from the total spin $S$ and the spins $S_A$, $S_B$ of the two pairs of opposite sites.

In [ ]:
def hamiltonian(repulsion):
    """Electrons hop around the ring, and sharing a site costs the repulsion."""
    return hopping + repulsion * shared_sites                                # [...] State <- State


def exchange(repulsion):
    """The exchange between neighbouring spins at strong repulsion."""
    return 4 * hopping_strength**2 / repulsion


# A sweep of the repulsion; the sixteen spin levels above the ground state, in units of the exchange.
samples = 157
spin_levels = 16
repulsion_sweep = hopping_strength * np.linspace(1.0, 40.0, samples)
heisenberg = np.array([1.0, 2.0, 3.0])
energies = hamiltonian(repulsion_sweep).eigvalsh()                           # [samples, modes] Scalar
excitations = (energies[:, :spin_levels] - energies[:, :1]) / exchange(repulsion_sweep)[:, None]   # [samples, spin levels] Scalar
render.draw_levels(repulsion_sweep / hopping_strength, excitations, heisenberg);

In [ ]:
print("the sixteen spin levels at the strongest repulsion, in units of the exchange:\n", excitations[-1].to_array())

## 3. A flipped spin walks the ring

Start with spin up on three sites and spin down on the fourth. Time acts on a state through the cosine and the sine of the Hamiltonian times time: each of the Hamiltonian's states keeps its share of the start and turns by its own energy, and the state after a while is a pair of real 4-vectors, its cosine part and its sine part. How many electrons an orbital holds is its number's expectation in both parts. Each repulsion runs for one cycle of its own exchange phase, the exchange times time. Below, each site's spin-up count as a red bar up and its spin-down count as a blue bar down. The flipped spin spreads to both neighbours, gathers on the opposite site halfway, and comes back. At strong repulsion it arrives there almost whole; at weaker repulsion the electrons also move as charge, and the transfer is blurred.

In the Heisenberg ring a single flipped spin reaches the opposite site with probability $\sin^4(J\tau/2)$, one at $J\tau = \pi$.

In [ ]:
def turn(energies, modes, start, times):
    """The cosine and the sine of the Hamiltonian times time applied to a start, from the Hamiltonian's
    states: each keeps its share of the start and turns by its own energy."""
    shares = modes * modes.reverse().scalar_product(start)                   # [..., modes] State
    angles = energies[..., None, :] * times[..., :, None]                    # [..., times, modes] Scalar
    return (shares[..., None, :] * angles.cos()).sum(axis=-1), (shares[..., None, :] * angles.sin()).sum(axis=-1)


# Three repulsions, one entry per case, each over one cycle of its exchange phase.
repulsions = hopping_strength * np.array([4.0, 8.0, 20.0])
frames = 151
frame_ms = 50
exchange_phase = np.linspace(0.0, 2 * np.pi, frames)
walk_energies, walk_modes = hamiltonian(repulsions).eigh()                   # [cases, modes] Scalar, State
times = exchange_phase / exchange(repulsions)[:, None]                       # [cases, times]
cosine_part, sine_part = turn(walk_energies, walk_modes, start, times)       # [cases, times] State each
# Each orbital's number, its expectation in both parts.
cosine_open, sine_open = cosine_part[..., None, None], sine_part[..., None, None]   # [cases, times, 1, 1] State each
occupation = cosine_open.reverse().scalar_product(numbers(cosine_open)) + sine_open.reverse().scalar_product(numbers(sine_open))   # [cases, times, sites, spins] Scalar
render.inline(render.animate_ring([f"$U/t = {ratio:g}$" for ratio in repulsions / hopping_strength], occupation), frame_ms)

In [ ]:
counts = occupation.to_array()                                               # [cases, times, sites, spins]
print("electrons, largest deviation from four:", np.abs(counts.sum(axis=(-1, -2)) - 4).max())
print("spin down on the opposite site halfway, per repulsion:", counts[:, frames // 2, 1, 1])
print("against the Heisenberg ring at the strongest, largest difference:", np.abs(counts[-1, :, 1, 1] - np.sin(exchange_phase / 2) ** 4).max())

In [ ]:
# checks
# The lift acts on each electron in turn; the shared sites count the doubly occupied sites; at strong
# repulsion the spin levels approach the Heisenberg ring's, the electrons stay four, and the flipped spin
# reaches the opposite site as the Heisenberg ring says.
np.testing.assert_allclose((hopping(start) - each_in_turn).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((shared_sites(mv.a ^ mv.b ^ mv.c ^ mv.d) - 2 * (mv.a ^ mv.b ^ mv.c ^ mv.d)).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(excitations[-1].to_array()[1:4], heisenberg[0], atol=0.05)
np.testing.assert_allclose(occupation.to_array().sum(axis=(-1, -2)), 4.0, atol=1e-11)
np.testing.assert_allclose(occupation.to_array()[-1, :, 1, 1], np.sin(exchange_phase / 2) ** 4, atol=0.05)